# EBM example 

In [ ]:
!pip install interpret

In [ ]:
# Allow visualisation of ebm results 
from interpret import set_visualize_provider 
from interpret.provider import InlineProvider 
set_visualize_provider(InlineProvider()) 

In [ ]:
# Import necessary packages 
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import roc_auc_score 
from interpret.glassbox import ExplainableBoostingClassifier 
from interpret import show 

In [ ]:
bankruptcy = pd.read_csv("Datasets/Bankruptcy.csv")

In [ ]:
def clean_column_names(df):
    
    df.columns = (
        df.columns
        .str.strip()                      # remove leading/trailing spaces
        .str.lower()                      # lowercase
        .str.replace(" ", "_")            # replace spaces with underscores
        .str.replace(r"[^\w\s]", "", regex=True)  # remove special characters
        .str.replace("__", "_")           # remove double underscores
    )
    
    return df

bankruptcy = clean_column_names(bankruptcy)

In [ ]:
bankruptcy1 = bankruptcy.drop(columns = ["net_income_flag"])

In [ ]:
X = bankruptcy1.drop(columns = ["bankrupt"])
y = bankruptcy1["bankrupt"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Make an ebm instance and fit it 
ebm = ExplainableBoostingClassifier() 
ebm.fit(X_train, y_train) 

In [ ]:
# See how well the ebm fitted the data 
auc = roc_auc_score(y_test, ebm.predict_proba(X_test)[:, 1]) 
print("AUC: {:.3f}".format(auc)) 

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predicted probabilities
y_prob = ebm.predict_proba(X_test)[:, 1]

# Predicted class labels
y_pred = ebm.predict(X_test)

# AUC
auc = roc_auc_score(y_test, y_prob)
print("AUC: {:.3f}".format(auc))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
# Have a look at ebm results and internal workings 
show(ebm.explain_global()) 

In [ ]:
# Look at term scores for Age 
ebm.term_scores_[0] 

In [ ]:
# Alter the scores as we’d like 
# ebm.term_scores_[0][62:64]=np.array([3,3]) 

In [ ]:
import numpy as np
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)

# =========================================================
# 3-fold CV for EBM
# =========================================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

auc_scores = []
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
conf_matrices = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Fit EBM on this fold
    ebm = ExplainableBoostingClassifier(random_state=42)
    ebm.fit(X_train, y_train)

    # Predictions
    y_prob = ebm.predict_proba(X_test)[:, 1]
    y_pred = ebm.predict(X_test)

    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    # Store results
    auc_scores.append(auc)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)
    conf_matrices.append(cm)

    # Print fold results
    print(f"\n=== Fold {fold} ===")
    print("Confusion Matrix:")
    print(cm)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1:", f1)
    print("ROC AUC:", auc)

# =========================================================
# Overall CV summary
# =========================================================
print("\n=== 3-Fold CV Summary ===")
print("AUC scores:", auc_scores)
print("Mean AUC:", np.mean(auc_scores))
print("Std AUC:", np.std(auc_scores))

print("\nAccuracy scores:", accuracy_scores)
print("Mean Accuracy:", np.mean(accuracy_scores))

print("\nPrecision scores:", precision_scores)
print("Mean Precision:", np.mean(precision_scores))

print("\nRecall scores:", recall_scores)
print("Mean Recall:", np.mean(recall_scores))

print("\nF1 scores:", f1_scores)
print("Mean F1:", np.mean(f1_scores))

In [ ]:
print("\n=== 3-Fold CV Summary ===")
print("AUC scores:", auc_scores)
print("Mean AUC:", np.mean(auc_scores))
print("Std AUC:", np.std(auc_scores))

print("\nAccuracy scores:", accuracy_scores)
print("Mean Accuracy:", np.mean(accuracy_scores))
print("Std AUC:", np.std(accuracy_scores))

print("\nPrecision scores:", precision_scores)
print("Mean Precision:", np.mean(precision_scores))
print("Std AUC:", np.std(precision_scores))

print("\nRecall scores:", recall_scores)
print("Mean Recall:", np.mean(recall_scores))
print("Std AUC:", np.std(recall_scores))

print("\nF1 scores:", f1_scores)
print("Mean F1:", np.mean(f1_scores))
print("Std AUC:", np.std(f1_scores))

In [ ]:
!pip install h2o
import h2o 
from h2o.automl import H2OAutoML 

In [ ]:
h2o.init() 
hf = h2o.H2OFrame(bankruptcy) 
train, test = hf.split_frame(ratios=[0.8], seed=1) 

In [ ]:
# Wait at least a year… 
aml = H2OAutoML(max_models=25, balance_classes=True, seed=1) 

# Train pipeline 
aml.train(y="bankrupt", training_frame=train) 

In [ ]:
# View leaderboard of models 
aml._leaderboard 

In [ ]:
print(aml._leaderboard)